# Olist Data Standardization & Lifecycle Pipeline

In [ ]:
import json
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path('__file__').resolve().parent.parent if '__file__' in locals() else Path.cwd()
RAW = BASE / 'Data_set'
CLEAN = BASE / 'processed' / 'cleaned'
CLEAN.mkdir(parents=True, exist_ok=True)

def standardize_text(value: object) -> object:
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')
    return ' '.join(text.split())

In [ ]:
customers = pd.read_csv(RAW / 'olist_customers_dataset.csv')
order_items = pd.read_csv(RAW / 'olist_order_items_dataset.csv')
order_payments = pd.read_csv(RAW / 'olist_order_payments_dataset.csv')
sellers = pd.read_csv(RAW / 'olist_sellers_dataset.csv')
translation = pd.read_csv(RAW / 'product_category_name_translation.csv')
products = pd.read_csv(RAW / 'olist_products_dataset.csv')
geolocation = pd.read_csv(RAW / 'olist_geolocation_dataset.csv')

orders = pd.read_csv(
    RAW / 'olist_orders_dataset.csv',
    parse_dates=[
        'order_purchase_timestamp', 'order_approved_at',
        'order_delivered_carrier_date', 'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
)

order_reviews = pd.read_csv(
    RAW / 'olist_order_reviews_dataset.csv',
    parse_dates=['review_creation_date', 'review_answer_timestamp']
)

In [ ]:
customers['customer_city_clean'] = customers['customer_city'].map(standardize_text)
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str).str.zfill(5)
customers['customer_state'] = customers['customer_state'].str.upper()

sellers['seller_city_clean'] = sellers['seller_city'].map(standardize_text)
sellers['seller_state'] = sellers['seller_state'].str.upper()
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)

orders['order_purchase_month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)
orders['order_purchase_year'] = orders['order_purchase_timestamp'].dt.year
orders['order_purchase_weekday'] = orders['order_purchase_timestamp'].dt.day_name()
orders['delivered_flag'] = orders['order_status'].eq('delivered').astype(int)

orders['late_delivery_flag'] = (orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']).astype('Int64')
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['approval_days'] = (orders['order_approved_at'] - orders['order_purchase_timestamp']).dt.total_seconds() / 86400
orders['carrier_days'] = (orders['order_delivered_carrier_date'] - orders['order_approved_at']).dt.total_seconds() / 86400

order_items['item_total_value'] = order_items['price'] + order_items['freight_value']
order_items['price_per_freight_ratio'] = order_items['price'] / order_items['freight_value'].replace(0, np.nan)
order_payments['payment_type'] = order_payments['payment_type'].str.lower()
order_reviews['has_review_text'] = order_reviews[['review_comment_title', 'review_comment_message']].notna().any(axis=1).astype(int)

translation['product_category_name'] = translation['product_category_name'].str.lower()
translation['product_category_name_english'] = translation['product_category_name_english'].str.lower()

products = products.merge(translation, on='product_category_name', how='left')
products['product_category_name'] = products['product_category_name'].str.lower()
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')
products['has_category'] = products['product_category_name'].notna().astype(int)
products['product_weight_kg'] = products['product_weight_g'] / 1000

geolocation_clean = (
    geolocation.assign(
        geolocation_city_clean=geolocation['geolocation_city'].map(standardize_text),
        geolocation_state=geolocation['geolocation_state'].str.upper(),
        geolocation_zip_code_prefix=geolocation['geolocation_zip_code_prefix'].astype(str).str.zfill(5),
    )
    .groupby(['geolocation_zip_code_prefix', 'geolocation_state'], as_index=False)
    .agg(
        geolocation_lat=('geolocation_lat', 'mean'),
        geolocation_lng=('geolocation_lng', 'mean'),
        geolocation_city_clean=('geolocation_city_clean', lambda s: s.mode().iat[0] if not s.mode().empty else s.iloc[0]),
    )
)

In [ ]:
order_revenue = order_items.groupby('order_id', as_index=False).agg(
    order_item_count=('order_item_id', 'count'),
    order_price=('price', 'sum'),
    order_freight=('freight_value', 'sum'),
    order_total_value=('item_total_value', 'sum'),
)

delivered_orders = orders[orders['delivered_flag'] == 1][
    ['order_id', 'customer_id', 'order_purchase_timestamp', 'delivery_days', 'late_delivery_flag']
].merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_city_clean']], 
    on='customer_id', how='left'
).merge(order_revenue, on='order_id', how='left')

customer_lifecycle = (
    delivered_orders.groupby('customer_unique_id', as_index=False)
    .agg(
        first_order_date=('order_purchase_timestamp', 'min'),
        last_order_date=('order_purchase_timestamp', 'max'),
        order_count=('order_id', 'nunique'),
        total_spent=('order_total_value', 'sum'),
        avg_order_value=('order_total_value', 'mean'),
        avg_delivery_days=('delivery_days', 'mean'),
        late_delivery_rate=('late_delivery_flag', 'mean'),
        state=('customer_state', 'first'),
        city=('customer_city_clean', 'first'),
    )
)

ref_date = orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
customer_lifecycle['recency_days'] = (ref_date - customer_lifecycle['last_order_date']).dt.days
customer_lifecycle['tenure_days'] = (customer_lifecycle['last_order_date'] - customer_lifecycle['first_order_date']).dt.days
customer_lifecycle['repeat_customer_flag'] = (customer_lifecycle['order_count'] > 1).astype(int)
customer_lifecycle['churn_flag_180d'] = (customer_lifecycle['recency_days'] > 180).astype(int)

In [ ]:
manifest = {
    'customers_clean.csv': customers,
    'orders_clean.csv': orders,
    'order_items_clean.csv': order_items,
    'order_payments_clean.csv': order_payments,
    'order_reviews_clean.csv': order_reviews,
    'products_clean.csv': products,
    'sellers_clean.csv': sellers,
    'geolocation_clean.csv': geolocation_clean,
    'product_category_name_translation_clean.csv': translation,
    'customer_lifecycle_features.csv': customer_lifecycle
}

for name, df in manifest.items():
    df.to_csv(CLEAN / name, index=False)

audit_summary = {
    'customers_rows': len(customers),
    'orders_rows': len(orders),
    'order_items_rows': len(order_items),
    'payments_rows': len(order_payments),
    'reviews_rows': len(order_reviews),
    'products_rows': len(products),
    'sellers_rows': len(sellers),
    'geolocation_rows': len(geolocation_clean),
    'customer_features_rows': len(customer_lifecycle),
    'delivered_orders': int(orders['delivered_flag'].sum()),
    'late_delivery_rate': round(float(orders[orders['delivered_flag'] == 1]['late_delivery_flag'].mean() * 100), 2),
}

with open(BASE / 'processed' / 'cleaning_audit.json', 'w', encoding='utf-8') as f:
    json.dump(audit_summary, f, indent=2, default=str)

print(json.dumps(audit_summary, indent=2))